# 17.4 Guidelines for piecewise-linear optimization

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

AMPL's piecewise-linear notation has the power to specify a variety of useful functions.
We summarize below its various forms, most of which have been illustrated earlier
in this chapter.

Because this notation is so general, it can be used to specify many functions that are
not readily optimized by any efficient and reliable algorithms. We conclude by describing
the kinds of piecewise-linear functions that are most likely to appear in tractable models,
with particular emphasis on the property of convexity or concavity.

**Forms for piecewise-linear expressions**

An AMPL piecewise-linear term has the general form

```ampl
<<breakpoint - list; slope - list>> pl - argument
```

where breakpoint-list and slope-list each consist of a comma-separated list of one or more
items. An item may be an individual arithmetic expression, or an indexing expression
followed by an arithmetic expression. In the latter case, the indexing expression must be
an ordered set; the item is expanded to a list by evaluating the arithmetic expression once
for each set member (as in the example of [Figure 17-3a](./17_1_cost_terms.ipynb#fig-17-3a)).

After any indexed items are expanded, the number of slopes must be one more than
the number of breakpoints, and the breakpoints must be nondecreasing. The resulting
piecewise-linear function is constructed by interleaving the slopes and breakpoints in the
order given, with the first slope to the left of the first breakpoint, and the last slope to the
right of the last breakpoint. By indexing breakpoints over an empty set, it is possible to
specify no breakpoints and one slope, in which case the function is linear.

The pl-argument may have one of the forms

```ampl
var - ref
(arg - expr)
(arg - expr, zero - expr)
```

The `var`-ref (a reference to a previously declared variable) or the arg-expr (an arithmetic
expression) specifies the point where the piecewise-linear function is to be evaluated.
The zero-expr is an arithmetic expression that specifies a place where the function is zero;
when the zero-expr is omitted, the function is assumed to be zero at zero.
**Suggestions for piecewise-linear models**

As seen in all of our examples, AMPL's terminology for piecewise-linear functions of
variables is limited to describing functions of individual variables. In model declarations,
no variables may appear in the breakpoint-list, slope-list and zero-expr (if any), while an
arg-expr can only be a reference to an individual variable. (Piecewise-linear expressions
in commands like display may use variables without limitation, however.)
A piecewise-linear function of an individual variable remains such a function when
multiplied or divided by an arithmetic expression without variables. AMPL also treats a
sum or difference of piecewise-linear and linear functions of the same variable as representing
one piecewise-linear function of that variable. A separable piecewise-linear
function of a model's variables is a sum or difference (using +, - or sum) of piecewiselinear
or linear functions of the individual variables. Optimizers can effectively handle
these separable functions, which are the ones that appear in our examples.

A piecewise-linear function is convex if successive slopes are nondecreasing (along
with the breakpoints), and is concave if the slopes are nonincreasing. The two kinds of
piecewise-linear optimization most easily handled by solvers are minimizing a separable
convex piecewise-linear function, and maximizing a separable concave piecewise-linear
function, `subject to` linear constraints. You can easily `check` that all of this chapter's
examples are of these kinds. AMPL can obtain solutions in these cases by translating to
an equivalent linear program, applying any LP solver, and then translating the solution
back; the whole sequence occurs automatically when you type solve.

Outside of these two cases, optimizing a separable piecewise-linear function must be
viewed as an application of integer programming — the topic of [Chapter 20](../20/20.md) — and
AMPL must translate piecewise-linear terms to equivalent integer programming forms.
This, too, is done automatically, for solution by an appropriate solver. Because integer
programs are usually much harder to solve than similar linear programs of comparable
size, however, you should not assume that just any separable piecewise-linear function
can be readily optimized; a degree of experimentation may be necessary to determine
how large an instance your solver can handle. The best results are likely to be obtained
by solvers that accept an option known (mysteriously) as "special ordered sets of type 2";
`check` the solver-specific documentation for details.

The situation for the constraints can be described in a similar way. However, a separable
piecewise-linear function in a constraint can be handled through linear programming
only under a restrictive set of circumstances:

- If it is convex and on the left-hand side of a ≤ constraint (or equivalently, the
right-hand side of a ≥ constraint);
- If it is concave and on the left-hand side of a ≥ constraint (or equivalently, the
right-hand side of a ≤ constraint).

Other piecewise-linearities in the constraints must be dealt with through integer programming
techniques, and the preceding comments for the case of the objective apply.

If you have access to a solver that can handle piecewise-linearities directly, you can
turn off AMPL's translation to the linear or integer programming form by setting the
option pl_linearize to 0. The case of minimizing a convex or maximizing a concave
separable piecewise-linear function can in particular be handled very efficiently by
piecewise-linear generalizations of LP techniques. A solver intended for nonlinear programming
may also accept piecewise-linear functions, but it is unlikely to handle them
reliably unless it has been specially designed for "nondifferentiable" optimization.

The differences between hard and easy piecewise-linear cases can be slight. This
chapter's transportation example is easy, in particular because the shipping rates increase
along with shipping volume. The same example would be hard if economies of scale
caused shipping rates to decrease with volume, since then we would be minimizing a concave
rather than a convex function. We cannot say definitively that shipping rates ought
to be one way or the other; their behavior depends upon the specifics of the situation
being modeled.

In all cases, the difficulty of piecewise-linear optimization gradually increases with
the total number of pieces. Thus piecewise-linear cost functions are most effective when
the costs can be described or approximated by relatively few pieces. If you need more
than about a dozen pieces to describe a cost accurately, you may be better off using a
nonlinear function as described in [Chapter 18](../18/18.md).

**Bibliography**

Robert Fourer, "A Simplex Algorithm for Piecewise-Linear Programming III: Computational
Analysis and Applications." Mathematical Programming 53 (1992) pp. 213-235. A survey of
conversions from piecewise-linear to linear programs, and of applications.

Robert Fourer and Roy E. Marsten, "Solving Piecewise-Linear Programs: Experiments with a
Simplex Approach." ORSA Journal on Computing 4 (1992) pp. 16-31. Descriptions of varied
applications and of experience in solving them.

Spyros Kontogiorgis, "Practical Piecewise-Linear Approximation for Monotropic Optimization."
INFORMS Journal on Computing 12 (2000) pp. 324-340. Guidelines for choosing the breakpoints
when approximating a nonlinear function by a piecewise-linear one.